# Topic: Document Chunking Strategies

## Definition (30-second explanation)
Think of a Large Language Model (LLM) as a brilliant intern with a very small short-term memory trying to read a 1,000-page encyclopedia. **Chunking** is the process of tearing that encyclopedia into perfectly sized, bite-sized paragraphs (chunks) and filing them in a cabinet (Vector Database) so the LLM only has to read the exact paragraph it needs to answer a specific question. 

## Why Interviewers Ask This
In Retrieval-Augmented Generation (RAG) pipelines, if your chunking is garbage, your retrieval is garbage. Interviewers want to see if you understand that chunking is not just "splitting text"; it is a massive architectural decision that dictates search accuracy, token costs, and whether context is destroyed before it ever reaches the LLM.

## Core Concepts (The 3-Layer Anatomy)
*   **The Bottleneck:** Embedding models (like OpenAI's `text-embedding-3-small`) have strict context limits (often 512 to 8192 tokens). If you pass a whole document, the embeddings lose granular detail and context is diluted. If chunks are too small, they lose the surrounding contextual meaning.
*   **The Mechanism:**
    *   **Fixed-size / Recursive Character:** Splits text strictly by token or character count. Recursive attempts to split on natural breaks (paragraphs `\n\n`, then sentences `\n`, then words) iteratively until the chunk is under the size limit.
    *   **Semantic:** Uses a lightweight embedding model to measure the mathematical distance between consecutive sentences. When the topic shifts (the distance crosses a threshold), it creates a split.
    *   **Agentic:** Passes the text to a smaller LLM and asks it to act as a human editor, intelligently grouping text into logical sections and attaching metadata summaries.
    *   **Chunk Overlap:** Duplicating the tail-end of Chunk A into the beginning of Chunk B (e.g., a 150-token overlap) so a crucial sentence or idea isn't sliced completely in half.
*   **The Trade-off:** Simple recursive chunking is lightning-fast and cheap but blindly cuts ideas in half. Semantic and Agentic chunking preserve complete thoughts perfectly but are computationally expensive, introducing heavy latency to the document ingestion pipeline.

## When to Use
*   **Recursive Character Chunking:** The industry default. Use it for standard documents, PDFs, or articles where paragraph breaks naturally separate ideas.
*   **Semantic Chunking:** When documents lack clear structure, or when paragraph boundaries frequently mix multiple distinct topics.
*   **Agentic Chunking:** For highly complex, unstructured, or high-value documents (e.g., medical journals, legal contracts) where preserving exact logical flow and metadata is worth the massive extra LLM API cost.

## Advantages
*   **Granular Retrieval:** Smaller, well-structured chunks allow the Vector Database to return the exact fact needed instead of a bloated, irrelevant chapter.
*   **Reduced Hallucinations:** Providing the LLM with dense, highly relevant context leaves less room for the model to guess or hallucinate.

## Limitations
*   **Loss of Global Context:** A chunk might say "He signed the bill," but the LLM won't know who "He" is if the pronoun's reference was left in the previous chunk.
*   **Cost Scaling:** Using Agentic or Semantic chunking on millions of documents can blow up your preprocessing budget and slow down data ingestion severely.

## Common Comparisons
*   **Fixed-size vs. Recursive:** Fixed-size splits blindly at exactly `N` characters (often breaking words in half). Recursive is smarter, trying to split on natural punctuation first.
*   **Recursive vs. Semantic:** Recursive uses hardcoded rules (punctuation). Semantic uses math (cosine similarity of embeddings) to group meanings.

## Common Interview Traps
*   **Forgetting Chunk Overlap:** If an interviewer asks how you prevent cutting sentences in half, the answer is *always* configuring a "Chunk Overlap" (usually 10-25% of the chunk size).
*   **Ignoring the Embedding Model Limit:** Candidates often say they will chunk at 2000 tokens, completely forgetting that their chosen embedding model might have a maximum sequence length of 512 tokens. 

## Python Syntax (LangChain)
```python
from langchain.text_splitter import RecursiveCharacterTextSplitter

# 1. Define the splitter with size and overlap
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,         # Maximum size of a chunk (in characters)
    chunk_overlap=200,       # Number of characters to overlap between chunks to preserve context
    length_function=len,     # How the length is measured
    separators=["\n\n", "\n", " ", ""]  # Tries to split by paragraph, then sentence, then word
)

# 2. Split the raw document text
raw_text = "Your massive document text goes here..."
chunks = text_splitter.split_text(raw_text)

# The result is a list of string chunks ready for embedding and vector DB insertion
```

## 45-Second Interview Answer
"In a RAG pipeline, chunking is how we prepare documents for embedding and retrieval. I default to Recursive Character chunking with a 10-20% overlap because it is fast, computationally cheap, and respects natural paragraph boundaries. If the text is highly unstructured and ideas span strange boundaries, I will upgrade to Semantic chunking, which uses an embedding model to group sentences by mathematical similarity. For extreme high-value use cases like complex legal documents, I might employ Agentic chunking, where an LLM orchestrates the grouping, but I avoid it for standard data pipelines due to the high preprocessing cost and ingestion latency."

## Practice Questions:

### Q1: Chunk Disconnects & Advanced Retrieval
**Question:** You used standard recursive chunking. A user asks to compare two concepts (Plan A vs. Plan B) that exist in separate chunks. The vector DB only retrieves Plan A. How does Agentic Chunking fix this, and if you can't afford Agentic Chunking, what advanced retrieval techniques solve it?

**Answer:**
1. **The Agentic/Metadata Fix:** Standard chunks lose global context. Agentic chunking fixes this by using an LLM to inject a metadata summary at the top of every chunk (e.g., *"This chunk is part of a pricing comparison detailing the Enterprise plan"*). When the user asks to "Compare pricing," the vector DB matches the query against that injected metadata, scoring both Plan A and Plan B chunks highly.
2. **The Advanced Retrieval Fix (No LLM Chunking):**
   * **Increase `top_k` & Re-rank:** Retrieve the top 10 chunks instead of 1, and use a Cross-Encoder Re-ranker to push the most relevant chunks to the top of the context window.
   * **Query Expansion:** Pass the user's query ("Compare A and B") to a fast, cheap LLM to generate multiple sub-queries ("What is A?", "What is B?"). Execute all sub-queries against the Vector DB, aggregate the results, and send them to the final LLM.
   * **Parent-Child Chunking (Small-to-Big):** Embed small, granular chunks (sentences) for highly accurate mathematical search, but structurally link them to a larger "Parent" chunk. If the vector search hits the child chunk, you pass the massive Parent chunk (containing both plans) to the LLM.

**Interview Tips:**
*   **Honesty Wins:** If you don't know a buzzword, diagnosing the underlying pipeline bottleneck (e.g., "The `top_k` must be too low") proves you have a real engineering mindset.
*   **Vocabulary:** Differentiate clearly between runtime techniques (Re-ranking, Query Expansion) and offline evaluation metrics (nDCG, MAP).